In [1]:
import pandas as pd
train_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_tcp_based\chunk-based-split-1\train_df_prepared.parquet", engine="pyarrow")
valid_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_tcp_based\chunk-based-split-1\valid_df_prepared.parquet", engine="pyarrow")
test_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_tcp_based\chunk-based-split-1\test_df_prepared.parquet", engine="pyarrow")

In [2]:
train_df["timestamp_num"] = pd.to_datetime(train_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
valid_df["timestamp_num"] = pd.to_datetime(valid_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
test_df["timestamp_num"] = pd.to_datetime(test_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9

In [3]:
train_df.sort_values(by='timestamp_num', inplace=True)
valid_df.sort_values(by='timestamp_num', inplace=True)
test_df.sort_values(by='timestamp_num', inplace=True)

### 1. Chuẩn bị dữ liệu Đồ thị (Graph Data Preparation) cho PyG
Chuyển đổi DataFrame thành cấu trúc `edge_index` và `edge_attr` của PyTorch Geometric.

In [4]:
import torch
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from torch_geometric.data import Data

print("--- BƯỚC 1: XỬ LÝ IP VÀ TẠO MAPPING ---")
all_ips = pd.concat([
    train_df['src_ip'], train_df['dst_ip'],
    valid_df['src_ip'], valid_df['dst_ip'],
    test_df['src_ip'], test_df['dst_ip']
]).unique()

ip_to_id = {ip: i for i, ip in enumerate(all_ips)}
num_nodes = len(all_ips)
print(f"Tổng số Nodes (IP duy nhất) trong toàn bộ dữ liệu: {num_nodes}")

def prepare_dynamic_graphs(df, ip_dict, window_size_sec=600):
    """Hàm CHUẨN MỰC: Cắt lược DataFrame thành mảng các Đồ thị theo Cửa sổ thời gian (Time-Windows)
       -> Giữ nguyên tính lịch sử thời gian, KO BỊ TEMPORAL LEAKAGE!"""
    min_time = df['timestamp_num'].min()
    
    # Tránh SettingWithCopyWarning
    df = df.copy()
    # Phân lô thời gian (VD: Cứ 600 giây thành 1 cục)
    df['time_window'] = ((df['timestamp_num'] - min_time) // window_size_sec).astype(int)
    
    dynamic_graphs = []
    cols_to_drop = ['flow_id', 'timestamp', 'timestamp_num', 'time_window', 'src_ip', 'dst_ip', 'protocol', 'src_port', 'dst_port', 'label']
    cols_to_drop = [c for c in cols_to_drop if c in df.columns]
    
    # Nhóm theo từng cửa sổ thời gian (Quá khứ -> Hiện tại -> Tương lai)
    for window_id, group in df.groupby('time_window'):
        if len(group) == 0:
            continue
            
        src_nodes = group['src_ip'].map(ip_dict).values
        dst_nodes = group['dst_ip'].map(ip_dict).values
        edge_index = torch.tensor(np.vstack([src_nodes, dst_nodes]), dtype=torch.long)
        
        edge_features = group.drop(columns=cols_to_drop).values
        edge_attr = torch.tensor(edge_features, dtype=torch.float32)
        
        labels = torch.tensor(group['label'].values, dtype=torch.long)
        
        # Bọc thành đối tượng PyG Data của KHUÔN HÌNH THỜI GIAN đó
        snapshot = Data(
            num_nodes=len(ip_dict), 
            edge_index=edge_index, 
            edge_attr=edge_attr, 
            y=labels
        )
        dynamic_graphs.append(snapshot)
        
    return dynamic_graphs

print("\n--- BƯỚC 2: CẮT ĐỒ THỊ THEO THỜI GIAN (TIME-WINDOW GRAPHS) ---")
WINDOW_SIZE = 600 # Nhóm từng khoảng 10 phút (600 giây) thành 1 đồ thị chuẩn
print(f"Đang tiến hành cắt đồ thị với cửa sổ {WINDOW_SIZE} giây / snapshot...")

train_graphs = prepare_dynamic_graphs(train_df, ip_to_id, WINDOW_SIZE)
valid_graphs = prepare_dynamic_graphs(valid_df, ip_to_id, WINDOW_SIZE)
test_graphs = prepare_dynamic_graphs(test_df, ip_to_id, WINDOW_SIZE)

print(f"-> Tập Train được cắt thành {len(train_graphs):,} Snapshots (Snapshot Đồ Thị Thời Gian)")
print(f"-> Tổng lưu lượng Edges trong Train: {sum([g.num_edges for g in train_graphs]):,}")

--- BƯỚC 1: XỬ LÝ IP VÀ TẠO MAPPING ---
Tổng số Nodes (IP duy nhất) trong toàn bộ dữ liệu: 49

--- BƯỚC 2: CẮT ĐỒ THỊ THEO THỜI GIAN (TIME-WINDOW GRAPHS) ---
Đang tiến hành cắt đồ thị với cửa sổ 600 giây / snapshot...
-> Tập Train được cắt thành 35 Snapshots (Snapshot Đồ Thị Thời Gian)
-> Tổng lưu lượng Edges trong Train: 7,139,054


### 2. Kiểm tra Đặc tính đồ thị (Graph Diagnostic Checks)
Dùng PyTorch Geometric `Data` object để bọc dữ liệu và tiến hành check tính hợp lệ của đồ thị (Label Imbalance, NaNs/Infs, Self-loops...)

In [5]:
from torch_geometric.data import Data

print("--- BƯỚC 3: KIỂM TRA ĐẶC TÍNH CỦA ĐỒ THỊ THỜI GIAN (DIAGNOSTIC CHECKS) ---")

# Lấy thử Snapshot đầu tiên và Snapshot bất kỳ có nhiều luồng nhất để phân tích
max_edge_graph = max(train_graphs, key=lambda g: g.num_edges)
first_graph = train_graphs[0]

print(f"1. Cấu trúc tổng quát của MỘT SỐ Đồ thị con (Snapshot lớn nhất):")
print(f"   - Số lượng Nút (IP): {max_edge_graph.num_nodes:,} (Khung IP cố định)")
print(f"   - Số lượng Cạnh (Flows): {max_edge_graph.num_edges:,}")
print(f"   - Đồ thị có chứa vòng lặp (Self-loops): {max_edge_graph.has_self_loops()}")
print(f"   - Đồ thị có Nút cô lập (Isolated Nodes)? {max_edge_graph.has_isolated_nodes()}")

print("\n2. Kiểm tra tính hợp lệ của Feature trên Snapshot đầu tiên:")
has_nan = torch.isnan(first_graph.edge_attr).any().item()
has_inf = torch.isinf(first_graph.edge_attr).any().item()
print(f"   - Có chứa giá trị NaN? {'CÓ LỖI' if has_nan else 'Không (An toàn)'}")
print(f"   - Có chứa giá trị Inf? {'CÓ LỖI' if has_inf else 'Không (An toàn)'}")

print("\n3. Kiểm tra tính Đa đồ thị (Multi-graph) trong Snapshot 10 phút nhỏ này:")
unique_edges = torch.unique(max_edge_graph.edge_index, dim=1).shape[1]
duplicate_edges = max_edge_graph.num_edges - unique_edges
print(f"   - Số cạnh trùng lặp trên cùng cặp IP: {duplicate_edges:,} / {max_edge_graph.num_edges:,} flows")
if duplicate_edges > 0:
    print("   -> Snapshot trong 10 phút là Multi-graph, mô phỏng đúng Network Burst (Bùng nổ băng thông)!")
    print("   -> (Giải pháp E-GraphSAGE dùng aggr='max' đã được chuẩn bị để quét các tín hiệu ẩn)")

print("\n4. Kiểm tra phân phối Nhãn (Class Imbalance) trên toàn bộ các Snapshot Train:")
total_label_counts = torch.zeros(13, dtype=torch.int64) 
total_edges = sum([g.num_edges for g in train_graphs])

for g in train_graphs:
    counts = torch.bincount(g.y, minlength=13)
    total_label_counts += counts

for i, count in enumerate(total_label_counts[:13]):
    if count > 0:
        percentage = (count.item() / total_edges) * 100
        print(f"   - Lớp {i}: {count.item():>9,} flows ({percentage:>6.2f}%)")

--- BƯỚC 3: KIỂM TRA ĐẶC TÍNH CỦA ĐỒ THỊ THỜI GIAN (DIAGNOSTIC CHECKS) ---
1. Cấu trúc tổng quát của MỘT SỐ Đồ thị con (Snapshot lớn nhất):
   - Số lượng Nút (IP): 49 (Khung IP cố định)
   - Số lượng Cạnh (Flows): 1,328,637
   - Đồ thị có chứa vòng lặp (Self-loops): False
   - Đồ thị có Nút cô lập (Isolated Nodes)? True

2. Kiểm tra tính hợp lệ của Feature trên Snapshot đầu tiên:
   - Có chứa giá trị NaN? Không (An toàn)
   - Có chứa giá trị Inf? Không (An toàn)

3. Kiểm tra tính Đa đồ thị (Multi-graph) trong Snapshot 10 phút nhỏ này:


   - Số cạnh trùng lặp trên cùng cặp IP: 1,328,616 / 1,328,637 flows
   -> Snapshot trong 10 phút là Multi-graph, mô phỏng đúng Network Burst (Bùng nổ băng thông)!
   -> (Giải pháp E-GraphSAGE dùng aggr='max' đã được chuẩn bị để quét các tín hiệu ẩn)

4. Kiểm tra phân phối Nhãn (Class Imbalance) trên toàn bộ các Snapshot Train:
   - Lớp 0:   533,000 flows (  7.47%)
   - Lớp 1: 1,328,637 flows ( 18.61%)
   - Lớp 2:   199,671 flows (  2.80%)
   - Lớp 3: 1,197,831 flows ( 16.78%)
   - Lớp 4:   596,900 flows (  8.36%)
   - Lớp 5:   507,000 flows (  7.10%)
   - Lớp 6: 1,493,878 flows ( 20.93%)
   - Lớp 7:   149,611 flows (  2.10%)
   - Lớp 8:   340,730 flows (  4.77%)
   - Lớp 9:     4,248 flows (  0.06%)
   - Lớp 10:     2,943 flows (  0.04%)
   - Lớp 11:    56,272 flows (  0.79%)
   - Lớp 12:   728,333 flows ( 10.20%)


In [6]:
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing

# 1. ĐỊNH NGHĨA LỚP MESSAGE PASSING KẾT HỢP CẠNH (EDGE-ENHANCED SAGE)
class EdgeSAGEConv(MessagePassing):
    def __init__(self, in_channels, out_channels, edge_channels, dropout=0.2):
        # TỐI ƯU 1: Đổi từ 'mean' sang 'max' pooling
        # Vì data bị vón cục (nhiều flow cùng 1 class lấn át), 'mean' sẽ cào bằng làm tàng hình class phụ.
        # 'max' pooling giúp GraphSAGE bắt được tín hiệu của các luồng dị biệt (cực kỳ hiệu quả cho Intrusion Detection)
        super().__init__(aggr='max') 
        
        self.msg_mlp = nn.Sequential(
            nn.Linear(in_channels + edge_channels, out_channels),
            nn.LayerNorm(out_channels),
            nn.ReLU(),
            nn.Dropout(dropout) # Thêm nhiễu để bớt học vẹt bối cảnh hàng xóm
        )
        
        self.update_mlp = nn.Sequential(
            nn.Linear(in_channels + out_channels, out_channels),
            nn.LayerNorm(out_channels),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

    def forward(self, x, edge_index, edge_attr):
        return self.propagate(edge_index, x=x, edge_attr=edge_attr)

    def message(self, x_j, edge_attr):
        msg_input = torch.cat([x_j, edge_attr], dim=-1)
        return self.msg_mlp(msg_input)

    def update(self, aggr_out, x):
        update_input = torch.cat([x, aggr_out], dim=-1)
        return self.update_mlp(update_input)

# 2. ĐỊNH NGHĨA MODEL TỔNG THỂ 
class EGraphSAGE(nn.Module):
    def __init__(self, num_nodes, node_emb_dim, edge_in_dim, hidden_dim, num_classes):
        super().__init__()
        self.node_emb = nn.Embedding(num_nodes, node_emb_dim)
        
        self.edge_norm = nn.LayerNorm(edge_in_dim) 
        
        # Thêm dropout nhẹ vào Convolution
        self.conv1 = EdgeSAGEConv(node_emb_dim, hidden_dim, edge_in_dim, dropout=0.1)
        self.conv2 = EdgeSAGEConv(hidden_dim, hidden_dim, edge_in_dim, dropout=0.1)
        
        classifier_dim = hidden_dim * 2 + edge_in_dim
        self.classifier = nn.Sequential(
            nn.Linear(classifier_dim, hidden_dim * 2),
            nn.LayerNorm(hidden_dim * 2),
            nn.ReLU(),
            nn.Dropout(0.4), # Tăng Dropout tầng cuối để chống Overfitting
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(hidden_dim, num_classes)
        )

    def encode_context(self, edge_index, edge_attr):
        clean_attr = torch.nan_to_num(edge_attr, nan=0.0, posinf=0.0, neginf=0.0)
        clean_attr = self.edge_norm(clean_attr)
        
        x = self.node_emb.weight
        x = F.relu(self.conv1(x, edge_index, clean_attr))
        x = self.conv2(x, edge_index, clean_attr)
        return x, clean_attr

    def predict_edges(self, x_context, clean_attr, edge_index, query_idx):
        src_nodes = edge_index[0, query_idx]
        dst_nodes = edge_index[1, query_idx]
        batch_attr = clean_attr[query_idx]
        
        edge_repr = torch.cat([x_context[src_nodes], x_context[dst_nodes], batch_attr], dim=-1)
        return self.classifier(edge_repr)

print("Đã xây dựng xong Mô Hình SAGE (Tối ưu hóa Aggregation & Dropout)!")

Đã xây dựng xong Mô Hình SAGE (Tối ưu hóa Aggregation & Dropout)!


In [7]:
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
import torch
import torch.nn as nn

print("--- BƯỚC 4: KHỞI TẠO MÔ HÌNH VÀ THUẬT TOÁN TỐI ƯU ---")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Đang sử dụng thiết bị: {device}")

node_emb_dim = 128
hidden_dim = 256  
edge_in_dim = train_graphs[0].edge_attr.shape[1]

num_classes = 13  # Cập nhật thành 13 lớp cho dữ liệu TCP-BASED

model = EGraphSAGE(
    num_nodes=len(ip_to_id),
    node_emb_dim=node_emb_dim,
    edge_in_dim=edge_in_dim,
    hidden_dim=hidden_dim,
    num_classes=num_classes
).to(device)

optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

print("\n[Tự động] Tính toán Class Weights...")
total_label_counts = torch.zeros(num_classes, dtype=torch.int64)
for g in train_graphs:
    counts = torch.bincount(g.y, minlength=num_classes)
    total_label_counts += counts

# Ngăn lỗi chia cho 0
total_label_counts = torch.where(total_label_counts == 0, torch.ones_like(total_label_counts), total_label_counts)

# Tính nghịch đảo tuyến tính gốc
weights = 1.0 / total_label_counts.float()

# TỐI ƯU 3: WEIGHT CLAMPING MỀM HƠN
# Để Dynamic Undersampling phát huy tác dụng, ta nới rộng tỉ lệ trần là 500 (thay vì 100)
min_weight = weights.min()
max_weight = min_weight * 50.0  
weights = torch.clamp(weights, min=min_weight, max=max_weight)

# Chuẩn hoá weights
weights = weights / weights.sum() * num_classes 
weights = weights.to(device)

print(" -> Trọng số (Class Weights - Clamped) đã tính xong:", weights.cpu().numpy())

# TỐI ƯU 4: LABEL SMOOTHING = 0.05
criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.05)

print("\nKhởi tạo hoàn tất! Cấu hình nâng cấp với CLAMPED WEIGHTS + LABEL SMOOTHING đã sẵn sàng huấn luyện.")

--- BƯỚC 4: KHỞI TẠO MÔ HÌNH VÀ THUẬT TOÁN TỐI ƯU ---
Đang sử dụng thiết bị: cuda

[Tự động] Tính toán Class Weights...
 -> Trọng số (Class Weights - Clamped) đã tính xong: [0.2248123  0.09018638 0.60011196 0.10003494 0.20074546 0.23634116
 0.08021068 0.8009101  0.35167128 4.010534   4.010534   2.1293888
 0.16451947]

Khởi tạo hoàn tất! Cấu hình nâng cấp với CLAMPED WEIGHTS + LABEL SMOOTHING đã sẵn sàng huấn luyện.


In [8]:
import copy
import time
import random
import numpy as np
from sklearn.metrics import classification_report

print("--- BƯỚC 5: HUẤN LUYỆN, EARLY STOPPING & ĐÁNH GIÁ (ANTI-NAN GRAPHS) ---")

def evaluate_graphs(model, graphs, criterion, device):
    """Hàm đánh giá Cấu trúc mới"""
    model.eval()
    total_loss = 0
    total_edges = sum(g.num_edges for g in graphs)
    all_preds = []
    all_trues = []
    
    with torch.no_grad():
        for g in graphs:
            if g.num_edges == 0:
                continue
                
            g_edge_idx = g.edge_index.to(device)
            g_edge_attr = g.edge_attr.to(device)
            g_y = g.y.to(device)
            
            # GỘP 1: Dọn rác và Xây Context
            x_ctx, clean_attr = model.encode_context(g_edge_idx, g_edge_attr)
            
            # GỘP 2: Chia mẻ nhỏ Predict Tránh bể RAM
            for i in range(0, g.num_edges, 100000): 
                idx = torch.arange(i, min(i+100000, g.num_edges), device=device)
                
                val_out = model.predict_edges(x_ctx, clean_attr, g_edge_idx, idx)
                loss = criterion(val_out, g_y[idx])
                
                total_loss += loss.item() * len(idx)
                all_preds.append(torch.argmax(val_out, dim=1).cpu().numpy())
                all_trues.append(g_y[idx].cpu().numpy())
                
    avg_loss = total_loss / (total_edges if total_edges > 0 else 1)
    
    if len(all_preds) > 0:
        y_pred = np.concatenate(all_preds)
        y_true = np.concatenate(all_trues)
        macro_f1 = classification_report(y_true, y_pred, output_dict=True, zero_division=0)['macro avg']['f1-score']
    else:
        macro_f1, y_pred, y_true = 0.0, [], []
        
    return avg_loss, macro_f1, y_true, y_pred

# THIẾT LẬP THAM SỐ
epochs = 60
batch_size = 65536  # Tăng chunk lên để lướt siêu nhanh
patience = 8        
EVAL_FREQ = 2       # [TĂNG TỐC] Chỉ đánh giá (Validation) sau mỗi 2 epoch

best_val_f1 = -1.0
best_model_wts = copy.deepcopy(model.state_dict())
patience_counter = 0
total_train_edges = sum(g.num_edges for g in train_graphs)

# [TĂNG TỐC] Dùng AMP (Automatic Mixed Precision) để giảm một nửa thời gian train trên GPU
scaler = torch.amp.GradScaler('cuda')

print(f"\n=> Bắt đầu huấn luyện {epochs} Epochs với KỸ THUẬT SIÊU TỐC (Manual Gradient Accumulation + AMP)...")
start_time = time.time()

for epoch in range(epochs):
    model.train()
    total_train_loss = 0
    
    random.shuffle(train_graphs)
    
    for graph_idx, g in enumerate(train_graphs):
        if g.num_edges == 0:
            continue
            
        g_edge_idx = g.edge_index.to(device)
        g_edge_attr = g.edge_attr.to(device)
        g_y = g.y.to(device)
        
        optimizer.zero_grad()
        
        # 1. CHỈ TÍNH BỐI CẢNH 1 LẦN DUY NHẤT VỚI FP16 AMP (SIÊU NHANH)
        with torch.amp.autocast('cuda'):
            x_ctx, clean_attr = model.encode_context(g_edge_idx, g_edge_attr)
        
        # 2. TÁCH RỜI ĐỒ THỊ ĐỂ CHẶN BACKWARD LAN SÂU XUỐNG DƯỚI
        x_ctx_det = x_ctx.detach().requires_grad_(True)
        clean_attr_det = clean_attr.detach().requires_grad_(True)
        
        perm = torch.randperm(g.num_edges, device=device)
        
        # 3. CHỈ BACKWARD CỤC BỘ DÀNH CHO PHẦN CLASSIFIER
        for i in range(0, g.num_edges, batch_size):
            idx = perm[i:i+batch_size]
            
            with torch.amp.autocast('cuda'):
                preds = model.predict_edges(x_ctx_det, clean_attr_det, g_edge_idx, idx)
                # Loss được chia tỉ lệ theo batch để mô phỏng chính xác Full-Batch Gradient
                loss = criterion(preds, g_y[idx]) * (len(idx) / g.num_edges)
            
            # Quản lý scale loss cho FP16
            scaler.scale(loss).backward() 
            total_train_loss += loss.item() * g.num_edges
            
        # Kiểm tra tính toàn vẹn Gradient
        gx = x_ctx_det.grad if x_ctx_det.grad is not None else torch.zeros_like(x_ctx_det)
        gc = clean_attr_det.grad if clean_attr_det.grad is not None else torch.zeros_like(clean_attr_det)
        
        # 4. GỘP TOÀN BỘ YÊU CẦU GRADIENT VÀ TRUYỀN NGƯỢC QUA GNN
        torch.autograd.backward(
            tensors=[x_ctx, clean_attr],
            grad_tensors=[gx, gc]
        )
        
        # Step optimizer thông qua scaler
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        scaler.step(optimizer)
        scaler.update()
            
    avg_train_loss = total_train_loss / total_train_edges
    current_lr = optimizer.param_groups[0]['lr']
    
    # [TĂNG TỐC] CHỈ CHẠY VALIDATION Ở CÁC EPOCH CHẴN
    if (epoch + 1) % EVAL_FREQ == 0 or epoch == 0:
        val_loss, val_f1, _, _ = evaluate_graphs(model, valid_graphs, criterion, device)
        scheduler.step(val_f1)
        
        print(f"Epoch {epoch+1:>2}/{epochs} | LR: {current_lr:.6f} | Train Loss: {avg_train_loss:.4f} | Val Loss: {val_loss:.4f} | Val F1: {val_f1:.4f}")
        
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_model_wts = copy.deepcopy(model.state_dict())
            patience_counter = 0
            print(f"   -> Epoch {epoch+1}: Kỷ lục F1 mới ({val_f1:.4f})! Đã lưu trọng số.")
        else:
            patience_counter += EVAL_FREQ
            if patience_counter >= patience:
                print(f"\n=> EARLY STOPPING kích hoạt ở Epoch {epoch+1} (Ngừng cải thiện sau {patience} epochs).")
                break
    else:
        print(f"Epoch {epoch+1:>2}/{epochs} | LR: {current_lr:.6f} | Train Loss: {avg_train_loss:.4f} | --- Skipped Validation ---")

print(f"\nQuá trình Training hoàn tất trong {time.time() - start_time:.2f} giây.")

print("\n--- BƯỚC 6: ĐÁNH GIÁ TRÊN TẬP TEST TEST (VỚI MODEL TỐT NHẤT) ---")
model.load_state_dict(best_model_wts)
test_loss, test_f1, y_test_true, y_test_pred = evaluate_graphs(model, test_graphs, criterion, device)

print(f"Test Loss: {test_loss:.4f} | Test Macro-F1: {test_f1:.4f}\n")
print(classification_report(y_test_true, y_test_pred, digits=4, zero_division=0))

--- BƯỚC 5: HUẤN LUYỆN, EARLY STOPPING & ĐÁNH GIÁ (ANTI-NAN GRAPHS) ---

=> Bắt đầu huấn luyện 60 Epochs với KỸ THUẬT SIÊU TỐC (Manual Gradient Accumulation + AMP)...
Epoch  1/60 | LR: 0.001000 | Train Loss: 5.7516 | Val Loss: 4.7502 | Val F1: 0.0142
   -> Epoch 1: Kỷ lục F1 mới (0.0142)! Đã lưu trọng số.
Epoch  2/60 | LR: 0.001000 | Train Loss: 4.4572 | Val Loss: 3.7857 | Val F1: 0.0316
   -> Epoch 2: Kỷ lục F1 mới (0.0316)! Đã lưu trọng số.
Epoch  3/60 | LR: 0.001000 | Train Loss: 3.7054 | --- Skipped Validation ---
Epoch  4/60 | LR: 0.001000 | Train Loss: 3.3463 | Val Loss: 2.9760 | Val F1: 0.1951
   -> Epoch 4: Kỷ lục F1 mới (0.1951)! Đã lưu trọng số.
Epoch  5/60 | LR: 0.001000 | Train Loss: 3.1430 | --- Skipped Validation ---
Epoch  6/60 | LR: 0.001000 | Train Loss: 3.1809 | Val Loss: 2.6512 | Val F1: 0.2879
   -> Epoch 6: Kỷ lục F1 mới (0.2879)! Đã lưu trọng số.
Epoch  7/60 | LR: 0.001000 | Train Loss: 2.7552 | --- Skipped Validation ---
Epoch  8/60 | LR: 0.001000 | Train Loss: 2